# Import the Spark Session

In [1]:
from pyspark.sql import SparkSession

# Initialize the spark session

In [4]:
spark = SparkSession.builder \
    .appName("Employee dataset analysis") \
    .master("local[*]").getOrCreate() 
# If you wanted to specify number of nodes to compute then mention - master("local[n]")
# n is number
# local: Runs with 1 core.
# local[n]: Runs with n cores.
# local[*]: Runs with as many cores as logical processors on the machine. 


26/05/10 12:27:22 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [5]:
spark

## Dataset used from Kaggle
# https://www.kaggle.com/datasets/rohitgrewal/hr-data-mnc

# Read data from the dataset folder locally

In [32]:
df = spark.read.csv("/home/labuser/Desktop/spark_setup/Spark/dataset/HR_Data.csv",header=True)

In [34]:
df

DataFrame[Unnamed: 0: string, Employee_ID: string, Full_Name: string, Department: string, Job_Title: string, Hire_Date: string, Location: string, Performance_Rating: string, Experience_Years: string, Status: string, Work_Mode: string, Salary_INR: string]

# EDA

In [9]:
df.columns

['Unnamed: 0',
 'Employee_ID',
 'Full_Name',
 'Department',
 'Job_Title',
 'Hire_Date',
 'Location',
 'Performance_Rating',
 'Experience_Years',
 'Status',
 'Work_Mode',
 'Salary_INR']

<b> To see the data you can use show() 

In [10]:
df.show(1)

+----------+-----------+-------------+----------+-----------------+----------+------------------+------------------+----------------+--------+---------+----------+
|Unnamed: 0|Employee_ID|    Full_Name|Department|        Job_Title| Hire_Date|          Location|Performance_Rating|Experience_Years|  Status|Work_Mode|Salary_INR|
+----------+-----------+-------------+----------+-----------------+----------+------------------+------------------+----------------+--------+---------+----------+
|         0| EMP0000001|Joshua Nguyen|        IT|Software Engineer|2011-08-10|Isaacland, Denmark|                 5|              14|Resigned|  On-site|   1585363|
+----------+-----------+-------------+----------+-----------------+----------+------------------+------------------+----------------+--------+---------+----------+
only showing top 1 row



<b> printshema() - As we mentioned the inferSchema as true while reading the data. The spark internally understands the data and assigns the datatype to it. </b>

In [11]:
df.printSchema() 

root
 |-- Unnamed: 0: integer (nullable = true)
 |-- Employee_ID: string (nullable = true)
 |-- Full_Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Job_Title: string (nullable = true)
 |-- Hire_Date: date (nullable = true)
 |-- Location: string (nullable = true)
 |-- Performance_Rating: integer (nullable = true)
 |-- Experience_Years: integer (nullable = true)
 |-- Status: string (nullable = true)
 |-- Work_Mode: string (nullable = true)
 |-- Salary_INR: integer (nullable = true)



# To get the summary statistics for your data - describe()

In [12]:
df.describe("salary_inr").show()

+-------+------------------+
|summary|        salary_inr|
+-------+------------------+
|  count|           2000000|
|   mean|    896887.7556635|
| stddev|402610.30443774525|
|    min|            300000|
|    max|           2999976|
+-------+------------------+



In [13]:
row_count = df.count()
print(row_count)

2000000


In [14]:
df.head(5) # to get the first row use first()

[Row(Unnamed: 0=0, Employee_ID='EMP0000001', Full_Name='Joshua Nguyen', Department='IT', Job_Title='Software Engineer', Hire_Date=datetime.date(2011, 8, 10), Location='Isaacland, Denmark', Performance_Rating=5, Experience_Years=14, Status='Resigned', Work_Mode='On-site', Salary_INR=1585363),
 Row(Unnamed: 0=1, Employee_ID='EMP0000002', Full_Name='Julie Williams', Department='Marketing', Job_Title='SEO Specialist', Hire_Date=datetime.date(2018, 3, 2), Location='Anthonyside, Costa Rica', Performance_Rating=2, Experience_Years=7, Status='Active', Work_Mode='On-site', Salary_INR=847686),
 Row(Unnamed: 0=2, Employee_ID='EMP0000003', Full_Name='Alyssa Martinez', Department='HR', Job_Title='HR Manager', Hire_Date=datetime.date(2023, 3, 20), Location='Port Christinaport, Saudi Arabia', Performance_Rating=1, Experience_Years=2, Status='Active', Work_Mode='On-site', Salary_INR=1430084),
 Row(Unnamed: 0=3, Employee_ID='EMP0000004', Full_Name='Nicholas Valdez', Department='IT', Job_Title='Software

In [15]:
df.first()

Row(Unnamed: 0=0, Employee_ID='EMP0000001', Full_Name='Joshua Nguyen', Department='IT', Job_Title='Software Engineer', Hire_Date=datetime.date(2011, 8, 10), Location='Isaacland, Denmark', Performance_Rating=5, Experience_Years=14, Status='Resigned', Work_Mode='On-site', Salary_INR=1585363)

In [16]:
df.tail(1)

[Row(Unnamed: 0=1999999, Employee_ID='EMP2000000', Full_Name='Angela Lambert', Department='HR', Job_Title='Talent Acquisition Specialist', Hire_Date=datetime.date(2020, 11, 11), Location='Morganchester, Canada', Performance_Rating=1, Experience_Years=4, Status='Active', Work_Mode='Remote', Salary_INR=993718)]

In [17]:
df.summary().show()

26/05/10 12:28:19 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+----------------+-----------+-------------+----------+--------------------+-------------------+------------------+----------------+----------+---------+------------------+
|summary|      Unnamed: 0|Employee_ID|    Full_Name|Department|           Job_Title|           Location|Performance_Rating|Experience_Years|    Status|Work_Mode|        Salary_INR|
+-------+----------------+-----------+-------------+----------+--------------------+-------------------+------------------+----------------+----------+---------+------------------+
|  count|         2000000|    2000000|      2000000|   2000000|             2000000|            2000000|           2000000|         2000000|   2000000|  2000000|           2000000|
|   mean|        999999.5|       NULL|         NULL|      NULL|                NULL|               NULL|         3.0001485|        5.010287|      NULL|     NULL|    896887.7556635|
| stddev|577350.413527175|       NULL|         NULL|      NULL|                NULL|           

In [27]:
from pyspark.sql.functions import col, count, when

null_count = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])

In [29]:
null_count.explain() # To check the physical plan and see the how the data is running

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[], functions=[count(CASE WHEN isnull(Unnamed: 0#17) THEN Unnamed: 0 END), count(CASE WHEN isnull(Employee_ID#18) THEN Employee_ID END), count(CASE WHEN isnull(Full_Name#19) THEN Full_Name END), count(CASE WHEN isnull(Department#20) THEN Department END), count(CASE WHEN isnull(Job_Title#21) THEN Job_Title END), count(CASE WHEN isnull(Hire_Date#22) THEN Hire_Date END), count(CASE WHEN isnull(Location#23) THEN Location END), count(CASE WHEN isnull(Performance_Rating#24) THEN Performance_Rating END), count(CASE WHEN isnull(Experience_Years#25) THEN Experience_Years END), count(CASE WHEN isnull(Status#26) THEN Status END), count(CASE WHEN isnull(Work_Mode#27) THEN Work_Mode END), count(CASE WHEN isnull(Salary_INR#28) THEN Salary_INR END)])
   +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=249]
      +- HashAggregate(keys=[], functions=[partial_count(CASE WHEN isnull(Unnamed: 0#17) THEN Unnamed: 0 END)

In [30]:
null_count.show()

+----------+-----------+---------+----------+---------+---------+--------+------------------+----------------+------+---------+----------+
|Unnamed: 0|Employee_ID|Full_Name|Department|Job_Title|Hire_Date|Location|Performance_Rating|Experience_Years|Status|Work_Mode|Salary_INR|
+----------+-----------+---------+----------+---------+---------+--------+------------------+----------------+------+---------+----------+
|         0|          0|        0|         0|        0|        0|       0|                 0|               0|     0|        0|         0|
+----------+-----------+---------+----------+---------+---------+--------+------------------+----------------+------+---------+----------+



In [31]:
data = df # data is read from the df variable and stored in data variable

# Result 1

Fetch the active employees with experience more than 10. 

In [42]:
df

DataFrame[Unnamed: 0: string, Employee_ID: string, Full_Name: string, Department: string, Job_Title: string, hire_date: date, Location: string, Performance_Rating: string, Experience_Years: string, Status: string, Work_Mode: string, Salary_INR: string]

In [43]:
df_active = df.filter((df.Status == "Active") & (df.Experience_Years >10))

In [45]:
df_active.count()

112769

## Change the date format to yyyy-MM-dd

In [35]:
from pyspark.sql.functions import to_date

In [36]:
df = df.withColumn("hire_date",to_date(col("hire_date"),"yyyy-MM-dd"))

In [37]:
df.select("hire_date").show(1)

+----------+
| hire_date|
+----------+
|2011-08-10|
+----------+
only showing top 1 row



In [38]:
data.select("hire_date").show(1)

+----------+
| hire_date|
+----------+
|2011-08-10|
+----------+
only showing top 1 row



In [39]:
print(df.printSchema())
print(data.printSchema())

root
 |-- Unnamed: 0: string (nullable = true)
 |-- Employee_ID: string (nullable = true)
 |-- Full_Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Job_Title: string (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- Location: string (nullable = true)
 |-- Performance_Rating: string (nullable = true)
 |-- Experience_Years: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Work_Mode: string (nullable = true)
 |-- Salary_INR: string (nullable = true)

None
root
 |-- Unnamed: 0: integer (nullable = true)
 |-- Employee_ID: string (nullable = true)
 |-- Full_Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Job_Title: string (nullable = true)
 |-- Hire_Date: date (nullable = true)
 |-- Location: string (nullable = true)
 |-- Performance_Rating: integer (nullable = true)
 |-- Experience_Years: integer (nullable = true)
 |-- Status: string (nullable = true)
 |-- Work_Mode: string (nullable = true)
 |-- Sala

# Transformation 1 